# PneumoFusionNet — Phase 2.1: Multimodal Fusion (Balanced Set)
## ResNet50+DSC+GCSA (Image) + Bio_ClinicalBERT (Text) → Binary Pneumonia Detection

### Architecture (Paper-Aligned)
```
CXR Image   ──►  [Frozen Phase 1: ResNet50+DSC+GCSA]  ──►  1024-d  ─┐
                                                                        ├──► Concat (1792-d)
Report Text ──►  [Bio_ClinicalBERT, fine-tune last 2 layers]  ──►  768-d  ─┘
                                                                        │
                              LayerNorm(1792) → 512 → GELU → 256 → GELU → 2
```

### Key Difference from Phase 2
> **Phase 2** used the full imbalanced dataset (118 Normal / 21 Pneumonia).  
> **Phase 2.1** uses a **balanced subset** (21 Normal / 21 Pneumonia) — matching the Phase 1.1 approach.  
> This eliminates class imbalance bias and provides a fair comparison.

### Paper Reference
> *"A multi-modal deep learning solution for precise pneumonia diagnosis: the PneumoFusion-Net model"*  
> *Frontiers in Physiology*, March 2025. DOI: 10.3389/fphys.2025.1512835

### Data Leakage Prevention
- **Exact same balanced sampling** as Phase 1.1 (SEED=42, 21+21=42 samples)
- Image encoder **fully frozen** — Phase 1 checkpoint, no gradient updates
- Normalization stats computed from **train set only**
- Model checkpoint selected by **best val AUC**, not train loss

---

## Author
IIT Guwahati · B.Sc. in Data Science & Artificial Intelligence  
📧 ay346185@gmail.com


## Step 1: Imports, Seeds & Device

In [ ]:
import os, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score, f1_score, auc
)
from collections import Counter

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA Memory:', torch.cuda.get_device_properties(DEVICE).total_memory / 1e9, 'GB')

## Step 2: Paths & Hyperparameters

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR     = r'C:\2026\PneumoFusionNet\mimic\mimic_pilot_139'
CSV_PATH     = os.path.join(BASE_DIR, 'dataset_139', 'mimic_multimodal_dataset_v3.csv')
PHASE1_MODEL = os.path.join(BASE_DIR, 'outputs', 'Phase_1.1(balanced set)', 'best_pneumofusion_mimic.pth')
SAVE_DIR     = os.path.join(BASE_DIR, 'outputs', 'Phase_2.1(balanced set)')
MODEL_PATH   = os.path.join(SAVE_DIR, 'best_phase2.1_multimodal.pth')
os.makedirs(SAVE_DIR, exist_ok=True)

# Image path correction — same OLD→NEW mapping as Phase 1 (must include \p10 to remove intermediate directory)
OLD_IMG = r'C:\2026\PneumoFusionNet\mimic\MIMIC_CXR_JPG_P1\p10'
NEW_IMG = r'C:\2026\PneumoFusionNet\mimic\mimic_pilot_139\mimic-cxr-jpg\2.1.0\files\p10'

# ── Hyperparameters ────────────────────────────────────────────────────────────
IMG_SIZE      = 224
BATCH_SIZE    = 8
NUM_EPOCHS    = 40
WARMUP_EPOCHS = 3         # Linear warmup epochs (same as Phase 1)
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 1.0       # Gradient clipping (same as Phase 1)
NUM_CLASSES   = 2
MAX_TEXT_LEN  = 128
BERT_MODEL    = 'emilyalsentzer/Bio_ClinicalBERT'
PATIENCE      = 10
CLASSES       = ['NORMAL', 'PNEUMONIA']

# Differential learning rates
LR_BERT   = 1e-5   # small — avoids catastrophic forgetting of MIMIC clinical knowledge
LR_FUSION = 5e-4   # larger — fusion MLP trains from scratch

print('BASE_DIR    :', BASE_DIR)
print('CSV exists  :', os.path.exists(CSV_PATH))
print('Phase1 ckpt :', os.path.exists(PHASE1_MODEL))
print('Save dir    :', SAVE_DIR)
print('BERT model  :', BERT_MODEL)

## Step 3: Load Multimodal Dataset & Fix Image Paths

In [ ]:
df = pd.read_csv(CSV_PATH)

# Fix image paths (same mapping as Phase 1)
df['image_path'] = df['image_path'].str.replace(OLD_IMG, NEW_IMG, regex=False)

# Verify images
n_found = df['image_path'].apply(os.path.exists).sum()
n_miss  = len(df) - n_found
print(f'Total samples     : {len(df)}')
print(f'Images found      : {n_found} / {len(df)}')
if n_miss > 0:
    print(f'WARNING: {n_miss} missing images!')
    print("  Sample path:", df[~df["image_path"].apply(os.path.exists)]['image_path'].iloc[0])
else:
    print('All images found!')

# ── Balanced sampling: 21 Normal + 21 Pneumonia (matches Phase 1.1) ──────────
normal_df    = df[df['label'] == 0].sample(n=21, random_state=SEED).reset_index(drop=True)
pneumonia_df = df[df['label'] == 1].sample(n=21, random_state=SEED).reset_index(drop=True)
df = pd.concat([normal_df, pneumonia_df]).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f'\nBalanced to {len(df)} samples (21 Normal + 21 Pneumonia)')

print('\nClass distribution:')
print(df['label_name'].value_counts())
print('\nFirst 5 rows:')
df.head()

## Step 4: Exact Same 70/15/15 Split as Phase 1

> **Why this prevents data leakage:**  
> Phase 2 is an extension of Phase 1. If we used a different split:
> - Samples that Phase 1 **tested on** could be in Phase 2's **train set** → unfair comparison
> - The Phase 1 checkpoint (used as image encoder here) would have "seen" Phase 2 test data → evaluation bias
>
> **Solution:** Use `SEED=42` + same `test_size=0.30 → 0.50` cascade to reproduce Phase 1's exact split.

In [ ]:
# Exactly matches Phase 1 split logic
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label']
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print('=' * 65)
print('SPLIT SUMMARY — Matching Phase 1 Exactly (SEED=42)')
print('=' * 65)
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    np_ = split['label'].sum()
    nn_ = (split['label'] == 0).sum()
    pct = len(split) / len(df) * 100
    print(f'{name:6s} | Total: {len(split):3d} ({pct:.0f}%) | Normal: {nn_:3d} | Pneumonia: {np_:3d}')
print('=' * 65)

## Step 5: Compute Normalization Stats from Train Set Only
> No leakage — same approach as Phase 1.

In [ ]:
print('Computing normalization stats from Phase 2 train set (no leakage)...')
means, stds = [], []
for idx in range(min(len(train_df), 200)):
    try:
        img = Image.open(train_df.iloc[idx]['image_path']).convert('L')
        arr = np.array(img, dtype=np.float32) / 255.0
        means.append(arr.mean())
        stds.append(arr.std())
    except:
        continue

MEAN = [float(np.mean(means))]
STD  = [float(np.mean(stds))]
print(f'Computed MEAN : {MEAN[0]:.4f}')
print(f'Computed STD  : {STD[0]:.4f}')

## Step 6: Image Transforms

In [ ]:
# Train: augmentation consistent with Phase 1
# Val/Test: deterministic inference only
tfms = {
    'train': transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((256, 256)),
        transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ]),
    'val': transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ]),
    'test': transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ]),
}
print('Transforms ready (dataset-specific normalization).')

## Step 7: Load Bio_ClinicalBERT Tokenizer

> **Upgrade from paper:** The paper uses BiLSTM+Attention for text encoding.  
> We use `emilyalsentzer/Bio_ClinicalBERT` — pretrained on MIMIC-III clinical notes,  
> the same hospital system as MIMIC-CXR. This gives superior understanding of  
> clinical abbreviations, anatomical terms, and radiological language.

In [ ]:
print(f'Loading tokenizer: {BERT_MODEL} ...')
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)

# Quick sanity check
enc = tokenizer(
    'No acute cardiopulmonary process.',
    return_tensors='pt',
    max_length=MAX_TEXT_LEN,
    truncation=True,
    padding='max_length'
)
print(f'Tokenizer loaded. Sample token count: {enc["input_ids"].shape[1]}')

## Step 8: Multimodal Dataset Class

In [ ]:
class MultimodalDataset(Dataset):
    """
    Returns:
        image          : Tensor [1, H, W]   — grayscale CXR
        input_ids      : Tensor [max_len]   — ClinicalBERT token IDs
        attention_mask : Tensor [max_len]
        label          : int  (0=Normal, 1=Pneumonia)
    """
    def __init__(self, df, tokenizer, transform=None, max_len=MAX_TEXT_LEN):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_len   = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── Image ──────────────────────────────────────────────────────────────
        img = Image.open(row['image_path'])
        if self.transform:
            img = self.transform(img)

        # ── Text (Read from report_path, use all except IMPRESSION) ──────────────
        report_path = row['report_path']
        text = "No clinical information reported."
        if os.path.exists(report_path):
            with open(report_path, "r", encoding="utf-8") as f:
                raw_text = f.read()
            import re
            parts = re.split(r'(?i)\bIMPRESSION\b', raw_text, maxsplit=1)
            if parts:
                text = parts[0].strip()
            else:
                text = raw_text.strip()

        enc  = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return (
            img,
            enc['input_ids'].squeeze(0),
            enc['attention_mask'].squeeze(0),
            int(row['label'])
        )


## Step 9: DataLoaders (Balanced — No Sampler Needed)

In [ ]:
# Classes are balanced (21:21) — use simple shuffle, no WeightedRandomSampler needed


dataloaders = {
    'train': DataLoader(
        MultimodalDataset(train_df, tokenizer, tfms['train']),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=0
    ),
    'val': DataLoader(
        MultimodalDataset(val_df, tokenizer, tfms['val']),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    ),
    'test': DataLoader(
        MultimodalDataset(test_df, tokenizer, tfms['test']),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    ),
}
print(f'Dataset sizes: {{train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}}}')

# Sanity check
img_b, ids_b, mask_b, lbl_b = next(iter(dataloaders['train']))
print(f'Batch — Image: {img_b.shape} | IDs: {ids_b.shape} | Mask: {mask_b.shape} | Labels: {lbl_b}')

## Step 10: Rebuild Phase 1 Image Encoder & Load Checkpoint

> The image encoder is **completely frozen** after loading — no gradients.  
> This ensures Phase 2 training only updates BERT and the fusion MLP.

In [ ]:
# ── Identical to Phase 1 architecture ─────────────────────────────────────────
class GCSA(nn.Module):
    """Global Channel-Spatial Attention — same as Phase 1"""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool     = nn.AdaptiveAvgPool2d(1)
        self.max_pool     = nn.AdaptiveMaxPool2d(1)
        self.mlp          = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels)
        )
        self.sigmoid      = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)

    def forward(self, x):
        B, C, H, W = x.shape
        avg    = self.avg_pool(x).view(B, C)
        mx     = self.max_pool(x).view(B, C)
        ch_att = self.sigmoid(self.mlp(avg) + self.mlp(mx)).view(B, C, 1, 1)
        x      = x * ch_att
        sp     = torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], dim=1)
        return x * self.sigmoid(self.conv_spatial(sp))


class DSC(nn.Module):
    """Depthwise Separable Convolution — same as Phase 1"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, 3, padding=1, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        return F.relu(self.bn(self.pw(self.dw(x))), inplace=True)


class PneumoFusionNet(nn.Module):
    """Phase 1 model — exact replica for loading Phase 1 weights."""
    def __init__(self, num_classes=2, freeze_until=7):
        super().__init__()
        resnet     = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        orig_conv1 = resnet.conv1
        resnet.conv1 = nn.Conv2d(1, 64, 7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            resnet.conv1.weight = nn.Parameter(
                orig_conv1.weight.mean(dim=1, keepdim=True)
            )
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        for i, child in enumerate(self.backbone.children()):
            if i < freeze_until:
                for p in child.parameters(): p.requires_grad = False
        self.dsc        = DSC(2048, 1024)
        self.gcsa       = GCSA(1024)
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(1024, 512),
            nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.pool(self.gcsa(self.dsc(self.backbone(x)))).flatten(1))

    def get_features(self, x):
        """Extract 1024-d embedding (no classifier head)."""
        return self.pool(self.gcsa(self.dsc(self.backbone(x)))).flatten(1)


# Load Phase 1 best checkpoint
print(f'Loading Phase 1 checkpoint: {PHASE1_MODEL}')
assert os.path.exists(PHASE1_MODEL), 'Phase 1 checkpoint missing! Run Phase 1 first.'

image_encoder = PneumoFusionNet().to(DEVICE)
image_encoder.load_state_dict(torch.load(PHASE1_MODEL, map_location=DEVICE))

# Freeze ALL parameters — no updates in Phase 2
for p in image_encoder.parameters():
    p.requires_grad = False
image_encoder.eval()

print('Phase 1 image encoder loaded and FULLY FROZEN.')
with torch.no_grad():
    d = torch.randn(2, 1, 224, 224).to(DEVICE)
    print(f'Image feature shape : {image_encoder.get_features(d).shape}')  # [2, 1024]

## Step 11: Multimodal Fusion Model

```
Image  ──►  [Frozen Phase 1]  ──►  1024-d  ─┐
                                              ├──► Concat(1792) ──► LayerNorm
Text   ──►  [ClinicalBERT]   ──►   768-d  ─┘      ──► Linear(512) + GELU + Dropout(0.4)
                                                   ──► Linear(256) + GELU + Dropout(0.3)
                                                   ──► Linear(2)
```

**BERT freezing strategy:**
- Freeze first 10 of 12 transformer layers
- Fine-tune only last 2 layers → LR=1e-5
- Preserves medical language knowledge, adapts final representations

In [ ]:
class MultimodalFusionNet(nn.Module):
    """
    Fuses:
      - Frozen image features (1024-d) from Phase 1 ResNet50+DSC+GCSA
      - ClinicalBERT [CLS] token (768-d) — last 2 layers fine-tuned
    Via concatenation + LayerNorm + GELU MLP → binary classification.
    """
    def __init__(self, bert_name=BERT_MODEL, img_dim=1024, txt_dim=768,
                 num_classes=NUM_CLASSES, freeze_bert_layers=10):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_name)

        # Freeze first `freeze_bert_layers` transformer layers
        for i, layer in enumerate(self.bert.encoder.layer):
            if i < freeze_bert_layers:
                for p in layer.parameters(): p.requires_grad = False

        fused_dim = img_dim + txt_dim  # 1792
        self.fusion = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, 512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def encode_text(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]  # [B, 768] — CLS token

    def forward(self, img_feats, input_ids, attention_mask):
        """img_feats: [B, 1024] — pre-extracted from frozen image encoder."""
        txt_feats = self.encode_text(input_ids, attention_mask)  # [B, 768]
        fused     = torch.cat([img_feats, txt_feats], dim=1)     # [B, 1792]
        return self.fusion(fused)                                 # [B, 2]

    def get_fused_features(self, img_feats, input_ids, attention_mask):
        """1792-d embedding for Phase 3 downstream tasks."""
        txt_feats = self.encode_text(input_ids, attention_mask)
        return torch.cat([img_feats, txt_feats], dim=1)


print(f'Loading ClinicalBERT: {BERT_MODEL}')
fusion_model = MultimodalFusionNet().to(DEVICE)

total     = sum(p.numel() for p in fusion_model.parameters())
trainable = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,} ({trainable/total*100:.1f}%)')

# Sanity check
with torch.no_grad():
    d_img  = torch.randn(2, 1024).to(DEVICE)
    d_ids  = torch.randint(0, 1000, (2, MAX_TEXT_LEN)).to(DEVICE)
    d_mask = torch.ones(2, MAX_TEXT_LEN, dtype=torch.long).to(DEVICE)
    out = fusion_model(d_img, d_ids, d_mask)
print(f'Output shape    : {out.shape}')  # [2, 2]

## Step 12: Loss, Optimizer & Scheduler

**Differential learning rates:**
- BERT layers: `lr=1e-5` — very small to avoid catastrophic forgetting of MIMIC clinical knowledge
- Fusion MLP: `lr=5e-4` — larger since it trains from scratch

**Scheduler:** Linear warmup (3 epochs) → CosineAnnealing — same as Phase 1 for consistency.

In [ ]:
# CrossEntropyLoss — Classes are balanced (21:21)
criterion = nn.CrossEntropyLoss()

# Differential learning rates
optimizer = optim.AdamW([
    {'params': fusion_model.bert.parameters(),   'lr': LR_BERT,   'weight_decay': WEIGHT_DECAY},
    {'params': fusion_model.fusion.parameters(), 'lr': LR_FUSION, 'weight_decay': WEIGHT_DECAY},
], betas=(0.9, 0.999))

# Linear warmup (3 epochs) then cosine annealing — same as Phase 1
linear_warmup = LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
cosine_anneal = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS)
scheduler     = SequentialLR(
    optimizer,
    schedulers=[linear_warmup, cosine_anneal],
    milestones=[WARMUP_EPOCHS]
)

print('Loss function : CrossEntropyLoss')
print('Optimizer     : AdamW (differential LR)')
print(f'  BERT layers : lr={LR_BERT}')
print(f'  Fusion MLP  : lr={LR_FUSION}')
print(f'Warmup epochs : {WARMUP_EPOCHS}')
print(f'Total epochs  : {NUM_EPOCHS}')
print(f'Gradient clip : {GRAD_CLIP}')
print(f'Early stop    : patience={PATIENCE} (on val AUC)')

## Step 13: Evaluation Helper

In [ ]:
def evaluate_multimodal(fusion_model, image_encoder, loader, criterion, device):
    """Run inference on loader, return metrics and raw outputs."""
    fusion_model.eval()
    image_encoder.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, ids, mask, labels in loader:
            imgs, ids, mask, labels = (
                imgs.to(device), ids.to(device), mask.to(device), labels.to(device)
            )
            img_feats = image_encoder.get_features(imgs)   # frozen
            outputs   = fusion_model(img_feats, ids, mask)
            loss      = criterion(outputs, labels)
            total_loss += loss.item() * len(labels)
            probs      = F.softmax(outputs, dim=1)[:, 1]
            all_preds.extend(outputs.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())

    avg_loss  = total_loss / len(loader.dataset)
    accuracy  = accuracy_score(all_labels, all_preds)
    auc_score = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    return avg_loss, accuracy, auc_score, all_preds, all_labels, all_probs


print('evaluate_multimodal() defined.')

## Step 14: Training Loop

Key design:
- Image encoder always in `eval()` — prevents BatchNorm/Dropout from updating
- Image features extracted with `torch.no_grad()` — saves GPU memory
- Gradient clipping `max_norm=1.0` — same as Phase 1
- Checkpoint saved on **best val AUC** (not train loss — avoids overfitting)
- Early stopping with `patience=10`

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}

best_val_auc     = 0.0
best_epoch       = 0
patience_counter = 0

print('\n' + '='*80)
print('PHASE 2 MULTIMODAL TRAINING')
print('='*80)

for epoch in range(NUM_EPOCHS):
    # ── Train ───────────────────────────────────────────────────────────────────
    fusion_model.train()
    image_encoder.eval()  # Always eval — it's frozen
    train_loss = train_correct = train_n = 0

    for imgs, ids, mask, labels in dataloaders['train']:
        imgs, ids, mask, labels = (
            imgs.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
        )
        optimizer.zero_grad()

        # Extract frozen image features (no_grad saves memory + time)
        with torch.no_grad():
            img_feats = image_encoder.get_features(imgs)

        outputs = fusion_model(img_feats, ids, mask)
        loss    = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(fusion_model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        train_loss    += loss.item() * len(labels)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_n       += len(labels)

    scheduler.step()
    train_loss_avg = train_loss / train_n
    train_acc      = train_correct / train_n

    # ── Validate ─────────────────────────────────────────────────────────────
    val_loss, val_acc, val_auc, _, _, _ = evaluate_multimodal(
        fusion_model, image_encoder, dataloaders['val'], criterion, DEVICE
    )

    history['train_loss'].append(train_loss_avg)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc     = val_auc
        best_epoch       = epoch
        patience_counter = 0
        torch.save(fusion_model.state_dict(), MODEL_PATH)
        marker = ' <- BEST'
    else:
        patience_counter += 1
        marker = ''

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1:2d}/{NUM_EPOCHS}] | '
              f'Train Loss: {train_loss_avg:.4f} ({train_acc:.3f}) | '
              f'Val Loss: {val_loss:.4f} ({val_acc:.3f}) | '
              f'Val AUC: {val_auc:.4f}{marker}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs)')
        break

print('='*80)
print(f'Training complete. Best val AUC: {best_val_auc:.4f} at epoch {best_epoch+1}')
print(f'Model saved to: {MODEL_PATH}')
print('='*80)

## Step 15: Training History Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Phase 2 — Multimodal Training History', fontsize=14, fontweight='bold')

axes[0].plot(history['train_loss'], label='Train', lw=2, alpha=0.8)
axes[0].plot(history['val_loss'],   label='Val',   lw=2, alpha=0.8)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['val_acc'], lw=2, color='green', label='Val Acc')
axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best epoch {best_epoch+1}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(history['val_auc'], lw=2, color='orange', label='Val AUC')
axes[2].axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best epoch {best_epoch+1}')
axes[2].axhline(best_val_auc, color='red', linestyle=':', alpha=0.5)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('AUC')
axes[2].set_title('Validation AUC')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'phase2_training_history.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Training history saved.')

## Step 16: Test Set Evaluation

In [ ]:
# Load best checkpoint (by val AUC)
fusion_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

test_loss, test_acc, test_auc, test_preds, test_labels, test_probs = evaluate_multimodal(
    fusion_model, image_encoder, dataloaders['test'], criterion, DEVICE
)

test_f1 = f1_score(test_labels, test_preds, average='macro')

print('\n' + '='*60)
print('PHASE 2 — TEST SET RESULTS')
print('='*60)
print(f'Loss      : {test_loss:.4f}')
print(f'Accuracy  : {test_acc:.4f}')
print(f'AUC       : {test_auc:.4f}')
print(f'F1 (macro): {test_f1:.4f}')
print('\nClassification Report:')
print(classification_report(test_labels, test_preds, target_names=CLASSES))
print('='*60)

## Step 17: Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Phase 2 — Multimodal Fusion Results', fontsize=14, fontweight='bold')

# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES,
            cbar_kws={'label': 'Count'}, ax=axes[0])
axes[0].set_xlabel('Predicted', fontsize=11)
axes[0].set_ylabel('Actual', fontsize=11)
axes[0].set_title('Confusion Matrix — Test Set')

# ROC Curve
fpr, tpr, _ = roc_curve(test_labels, test_probs)
roc_auc_val = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2.5, label=f'Phase 2 AUC = {roc_auc_val:.4f}')
axes[1].plot([0, 1], [0, 1], 'navy', lw=1.5, linestyle='--', label='Random')
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve — Test Set')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'phase2_results.png'), dpi=120, bbox_inches='tight')
plt.show()

## Step 18: Phase 1 vs Phase 2 Comparison

> **Fill in the Phase 1 results below** after Phase 1 training completes (from Step 12 of Phase 1 notebook).

In [ ]:
# ── Phase 1.1 (Balanced) results for comparison ───────────────────────────────
PHASE1_1_AUC = 0.9167
PHASE1_1_ACC = 0.8571
PHASE1_1_F1  = 0.8571

# Phase 2 (Unbalanced) results
PHASE2_AUC = 0.7037
PHASE2_ACC = 0.8095
PHASE2_F1  = 0.4474
# ─────────────────────────────────────────────────────────────────────────────────

test_f1 = f1_score(test_labels, test_preds, average='macro', zero_division=0)

print('=' * 75)
print('  PILOT STUDY — ALL PHASES COMPARISON (SEED=42)')
print('=' * 75)
print(f'{"Model":<40} {"AUC":>8} {"Accuracy":>10} {"F1 (macro)":>12}')
print('-' * 75)
print(f'{"Phase 1    (Image Only, Imbalanced)":<40} {0.6667:>8.4f} {0.7619:>10.4f} {0.4300:>12.4f}')
print(f'{"Phase 1.1  (Image Only, Balanced)":<40} {PHASE1_1_AUC:>8.4f} {PHASE1_1_ACC:>10.4f} {PHASE1_1_F1:>12.4f}')
print(f'{"Phase 2    (Image+Text, Imbalanced)":<40} {PHASE2_AUC:>8.4f} {PHASE2_ACC:>10.4f} {PHASE2_F1:>12.4f}')
print(f'{"Phase 2.1  (Image+Text, Balanced) ★":<40} {test_auc:>8.4f} {test_acc:>10.4f} {test_f1:>12.4f}')
print('=' * 75)

# Improvement over Phase 1.1
delta_auc = test_auc - PHASE1_1_AUC
delta_acc = test_acc - PHASE1_1_ACC
print(f'\n  Phase 2.1 vs Phase 1.1 (Balanced):')
print(f'    AUC improvement : {delta_auc:+.4f}')
print(f'    Acc improvement : {delta_acc:+.4f}')

# Bar chart — all 4 phases
fig, ax = plt.subplots(figsize=(12, 6))
metrics = ['AUC', 'Accuracy', 'F1 (macro)']
p1_vals   = [0.6667, 0.7619, 0.4300]
p11_vals  = [PHASE1_1_AUC, PHASE1_1_ACC, PHASE1_1_F1]
p2_vals   = [PHASE2_AUC, PHASE2_ACC, PHASE2_F1]
p21_vals  = [test_auc, test_acc, test_f1]

x = np.arange(len(metrics))
w = 0.2
bars1 = ax.bar(x - 1.5*w, p1_vals,  w, label='Phase 1 (Imbalanced)',   color='#6C8EBF', alpha=0.85)
bars2 = ax.bar(x - 0.5*w, p11_vals, w, label='Phase 1.1 (Balanced)',    color='#82B366', alpha=0.85)
bars3 = ax.bar(x + 0.5*w, p2_vals,  w, label='Phase 2 (Imbalanced)',    color='#D6B656', alpha=0.85)
bars4 = ax.bar(x + 1.5*w, p21_vals, w, label='Phase 2.1 (Balanced) ★',  color='#E07B39', alpha=0.85)

for bars in [bars1, bars2, bars3, bars4]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_ylim(0, 1.15)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Pilot Study — All Phases Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'all_phases_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()


## Step 19: Save Fused Features for Phase 3

> Split-wise extraction — no augmentation, no shuffling.  
> These 1792-d embeddings will be used in Phase 3 (structured clinical data from MIMIC-IV).

In [ ]:
print('\nExtracting fused features for Phase 3 (split-wise, no leakage)...')
fusion_model.eval()
image_encoder.eval()

for split_name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    split_loader = DataLoader(
        MultimodalDataset(split_df, tokenizer, tfms['val']),  # val transform (no aug)
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )
    feats, labs = [], []

    with torch.no_grad():
        for imgs, ids, mask, labels in split_loader:
            imgs, ids, mask = imgs.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE)
            img_feats  = image_encoder.get_features(imgs)
            fused_feat = fusion_model.get_fused_features(img_feats, ids, mask).cpu()
            feats.append(fused_feat)
            labs.extend(labels.tolist())

    fused_features = torch.cat(feats)  # [N, 1792]
    feat_path = os.path.join(SAVE_DIR, f'fused_features_{split_name}.pt')
    torch.save({'features': fused_features, 'labels': torch.tensor(labs)}, feat_path)

    meta_df          = split_df[['subject_id', 'study_id', 'label', 'label_name']].copy()
    meta_df['split'] = split_name
    meta_df.to_csv(feat_path.replace('.pt', '_meta.csv'), index=False)

    print(f'{split_name:6s} | Fused features: {fused_features.shape} | Saved to: {feat_path}')

print('\nPhase 2 COMPLETE.')
print('Fused features (1792-d) ready for Phase 3 (MIMIC-IV labs fusion).')
print(f'\nAll outputs saved to: {SAVE_DIR}')
for f in sorted(os.listdir(SAVE_DIR)):
    print(f'  - {f}')

## Summary

| Component | Detail |
|---|---|
| **Image Encoder** | Frozen Phase 1 ResNet50+DSC+GCSA → 1024-d |
| **Text Encoder** | Bio_ClinicalBERT [CLS] → 768-d (last 2 layers fine-tuned) |
| **Fusion** | Concat(1792) → LayerNorm → GELU MLP (512→256→2) |
| **Text Input** | IMPRESSION section from radiology report |
| **Dataset** | **Balanced: 21 Normal + 21 Pneumonia = 42 samples** |
| **Split** | 70/15/15 (SEED=42) — same as Phase 1.1 |
| **BERT LR** | 1e-5 (preserves MIMIC-III clinical knowledge) |
| **Fusion MLP LR** | 5e-4 (trains from scratch) |
| **Scheduler** | Linear warmup (3 ep) → CosineAnnealing |
| **Checkpoint** | Best **val AUC** (prevents overfitting) |
| **Data Leakage** | None — frozen encoder + same split + train-only norm stats |

---

## 🎯 Pilot Testing Completed — Key Findings

### ✅ What Was Accomplished
The **PneumoFusionNet pilot study** has been successfully completed across all phases:

| Phase | Model | Dataset | Key Result |
|-------|-------|---------|------------|
| **Phase 1** | Image Only (ResNet50+DSC+GCSA) | 139 samples (Imbalanced) | Baseline established |
| **Phase 1.1** | Image Only (ResNet50+DSC+GCSA) | 42 samples (Balanced) | AUC = 0.9167 |
| **Phase 2** | Image + Text (BioClinicalBERT) | 139 samples (Imbalanced) | AUC = 0.7037 |
| **Phase 2.1** | Image + Text (BioClinicalBERT) | 42 samples (Balanced) | To be evaluated |

### 🔑 Important Observations for Full-Scale Study

**1. Increase the Number of Samples**
> The current pilot uses only **139 total samples** (42 in balanced experiments).  
> This is far too small for a deep learning model with **134M+ parameters** to generalize well.  
> The full MIMIC-CXR dataset contains **~377,000 images** — scaling up is essential for reliable conclusions.

**2. Standardize Image Projections (PA/AP/Lateral Issue)**
> Currently, **all available images per study** are included — PA (posteroanterior), AP (anteroposterior), and sometimes lateral views.  
> These projections look **significantly different** from each other, which may confuse the model during training.  
> **Recommendation:** Filter to PA-only (or AP-only) images for consistency, or add projection type as an additional feature.

**3. Patient-Level Splitting Required**
> Some patients have **multiple studies/images** in the dataset.  
> The current row-level `train_test_split` may leak patient-specific anatomy across splits.  
> **Recommendation:** Use `GroupShuffleSplit` or `StratifiedGroupKFold` grouped by `subject_id` to prevent patient-level data leakage.

**4. Text Input Considerations**
> Using the **Impression** section provides strong signal but risks **target leakage** (the impression often contains the diagnosis).  
> For a clinically realistic model, consider using only the **Indication/History** section (available before diagnosis).  
> Alternatively, carefully validate that the text modality adds value beyond simply reading the label from the impression.

**5. Class Imbalance Strategy**
> The balanced subset (21:21) dramatically improves Pneumonia recall compared to the imbalanced set (118:21).  
> For the full-scale study, consider a combination of **oversampling + augmentation** rather than pure downsampling to preserve data.

**6. Work on a Perfect Dataset**
> Before scaling up, curate a **clean, well-structured dataset** with:  
> - Consistent image projection (PA only)  
> - One image per study  
> - Verified labels from CheXpert labeler  
> - Patient-level deduplication  
> - Matched MIMIC-IV lab values for Phase 3

---

### 🚀 Next Steps (Full-Scale Study)
1. Curate a clean MIMIC-CXR dataset (PA-only, one image/study, patient-level split)
2. Scale to 1000+ balanced samples per class
3. **Phase 3:** Add structured MIMIC-IV lab values (WBC, CRP, Procalcitonin) as 3rd modality
4. GradCAM visualizations + BERT attention heatmaps
5. Cross-validation for robust performance estimation
